In [ ]:
import io, h5py, zipfile
from scipy.io import savemat
from scipy import signal
import numpy as np
from tqdm import tqdm
import os, sys
sys.path.append(os.path.abspath(".."))
import mienc.support as ms
from mienc import Corrector, NonLinearEstimator

bands={i:b for i, b in enumerate([[ 1.,  4.], [ 4.,  8.], [ 8., 12.], [12., 15.], [15., 18.], [18., 30.], [30., 44.], [12., 30.], [ 1., 40.]], start=1)}

basePath = "NonLinearityData/EEG_el_so_BLP_NEW"
f_sample = 1000
def fs_band(band):
    return int(bands[band][1]*2*1.125+0.5)
fixTime = [[np.empty((1120//i, 1465, 50)) for __ in bands] for i in [1,2,4]]
fixSamples = [[np.empty((280, 1465, 50)) for __ in bands] for i in [1,2,4]]

zip_file = zipfile.PyZipFile("../.././NonLinearityData/Export_Giulio_Source_PCA_20231109_orig_shadow.zip")
zip_file.filelist    

In [ ]:
archive = h5py.File(f"EEG_el_so_BLP_PCA_20231109/Sub_1/source_ori.mat")

In [ ]:
archive[archive[archive['source_ori'][0,1]][0,0]][:,0]

In [ ]:
archive2 = h5py.File(f"EEG_el_so_BLP_PCA_20231109/Sub_1/source_mom3.mat")

In [ ]:
dipoles = [np.transpose(archive2[archive2[archive2['source_mom3'][0,i]][0,0]],(1,0,2)) for i in range(9)]

In [ ]:
%matplotlib widget
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
band_names = [r"$\delta$",r"$\theta$",r"$\alpha$",r"$\beta_{LOW}$",r"$\beta_{MID}$",r"$\beta_{HIGH}$",r"$\gamma$",r"$\beta_{ALL}$",r"$1-40 Hz$"]
fig, ax = plt.subplots(figsize=(15,16))
ax.axis("off")
for i in range(9):
    ax = fig.add_subplot(3,3,i+1,projection='3d')
    ax.plot3D(*dipoles[i][:,:,200])
    plt.title(band_names[i])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
plt.show()

In [ ]:
dipoles[band].shape

In [ ]:
from sklearn.decomposition import PCA
from  tqdm  import tqdm
import multiprocessing as mp

def ratio_band (dipole):
    return PCA().fit(dipole).explained_variance_ratio_[0]
# dip2 = [dipoles[band]/np.std(dipoles[band], axis=0) for band in range(9)]
dip3 = []
for band in tqdm(range(9), desc="normalise"):
    tmp = np.empty_like(dipoles[band])
    for i in range(1465):
        for j in range(3):
            tmp[:,j,i] = ms.normalise(dipoles[band][:,j,i])
    dip3.append(tmp[:,:,:])
ratios = []
with mp.Pool(23) as polla:
    for band in tqdm(range(9), desc="band"):
        ratios.append(np.array(polla.map(ratio_band, [dip3[band][:,:,i] for i in range(1465)], 20)))

In [ ]:
from sklearn.decomposition import PCA
from  tqdm  import tqdm
import multiprocessing as mp

ratios = []
with mp.Pool(23) as polla:
    for band in tqdm(range(9), desc="band"):
        ratios.append(np.zeros(1465))
        for i in tqdm(range(1465), desc="source"):
            ratios[band][i]=PCA().fit(dipoles[band][:,:,i].T).explained_variance_ratio_[0]

In [ ]:
ratios[0][:10]

In [ ]:
for i in range(9):
    print(np.linalg.norm(archive[archive[archive['source_ori'][0,i]][0,0]][:,0]- PCA().fit(dipoles[i][:,:,0].T).components_[0]))

In [ ]:
i=2

In [ ]:
np.var(np.dot(dipoles[i][:,:,0].T,archive[archive[archive['source_ori'][0,i]][0,0]][:,0]))/np.sum(np.var(dipoles[i][:,:,0], axis=1))

In [ ]:
PCA().fit(dipoles[i][:,:,0].T).explained_variance_ratio_[0]

In [ ]:
dipoles[0].shape

In [ ]:
exp = np.zeros(1465)
for i in range(1465):
    t = (np.random.random(8550)-0.5)*np.pi
    f = np.random.random(8550)*2*np.pi
    r = np.random.lognormal(1,1,8550)
    booo = np.zeros((8550, 3))
    booo[:,0]=r*np.cos(t)*np.cos(f)
    booo[:,1]=r*np.cos(t)*np.sin(f)
    booo[:,2]=r*np.sin(t)
    booo/=np.std(booo,0)
    exp[i] = PCA().fit(booo[:,:]).explained_variance_ratio_[0]


In [ ]:
plt.figure()
plt.hist(exp, "auto")
plt.title("faKE")
ax.set_xlabel('Explained variance ratio')
ax.set_ylabel('count')
ax.set_xlim(left=0.333)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15,16))
ax.axis("off")
ax = fig.add_subplot(1,1,1,projection='3d')
ax.plot3D(*booo.T)

In [ ]:
PCA(n_components=1).fit(dipoles[i][:,:,0].T).explained_variance_ratio_[0]

In [ ]:
%timeit PCA().fit(dipoles[1][:,:,0].T).explained_variance_ratio_[0]

In [ ]:
fig, ax = plt.subplots(figsize=(15,16))
ax.axis("off")
for band in tqdm(range(9), desc="band"):
    ax = fig.add_subplot(3,3,band+1)
    ax.hist(ratios[band], "auto")
    plt.title(band_names[band])
    ax.set_xlabel('Explained variance ratio')
    ax.set_ylabel('count')
    ax.set_xlim(left=0.333)
plt.show()

In [ ]:
zip_file.extract(f"EEG_el_so_BLP_PCA_20231109/Sub_1/source_mom3.mat")

In [ ]:
from scipy.io import loadmat

loadmat("../../NonLinearityData/EEG_el_so_BLP_NEW/NEW_EEG_fixedTime_band7.mat")["EEG"].shape